---
title: "Data Collection"
format:
    html: 
        code-fold: false
---

{{< include overview.qmd >}} 

{{< include methods.qmd >}} 

# Introduction and Motivation

The primary objective of this project is to construct a high-fidelity, multi-source dataset that correlates **grid-side power generation dynamics** with **environmental-side meteorological conditions**. specifically, we aim to:

1. **Aggregate** high-frequency (5-minute interval) wind power generation data from the Bonneville Power Administration (BPA), which serves as the "ground truth" for grid status.
2. **Integrate** granular weather observation data (wind speed, direction, etc.) from key NOAA/NWS stations across the Pacific Northwest (PDX, GEG, SEA, EUG).
3. **Synchronize** these two disparate data streams onto a unified temporal axis to enable the detection of "Ramp Events"—sudden, drastic fluctuations in wind power output.

By the end of this section, we will get one raw weather dataset csv (`BPA_Weather_IEM_Parallel.csv`) and four raw Wind Gen dataset(`WindGenTotalLoadYTD_202x.xlsx`).

# Overview of the Method

At a high level, the data collection workflow proceeds as follows:

(1) raw BPA generation reports are manually downloaded and normalized into a unified time-series format;

(2) Weather observations are programmatically retrieved from the IEM ASOS API and reshaped into a station-wise wide table;

(3) both data streams are aligned on a common UTC timestamp and merged to produce a consolidated dataset for downstream analysis.


### 1. Tools Used

The data collection pipeline was implemented entirely in Python within this Notebook, leveraging the following key libraries:

- **`pandas`:** The core engine for data manipulation. It was used for reading Excel/CSV files, performing complex "Pivot" operations to reshape weather data, and executing high-precision time-series merges (`merge_asof` / `join`).
- **`openpyxl`:** Essential for reading the `.xlsx` formatted reports provided by BPA.
- **`os` & `glob`:** Used for file system navigation to programmatically iterate through annual datasets (2022-2025) and automate the concatenation process.
- **IEM ASOS API:** An HTTP-based interface used to retrieve historical METAR weather data from the Iowa Environmental Mesonet.

### 2. Data Structure and Format

The project integrates two distinct data streams, each with its own native structure:

**Stream A: Grid Generation Data (BPA)———Manual Download from source link**

- **Source Link:** https://transmission.bpa.gov/Business/Operations/Wind/
- **Source Format:** Annual Excel (`.xlsx`) files.
- **Processed Structure:** We normalized these into a single **Long-Format Time Series**.
  - **Index:** `Date/Time (UTC)` (5-minute frequency).
  - **Columns:** 
    - **Date/Time (UTC)**: The timestamp in Coordinated Universal Time. We use this standard time to perfectly match the grid data with the weather data.
    - **Date/Time**: The local time in the Pacific Northwest (Pacific Standard/Daylight Time).
    - **TOTAL WIND GENERATION (SCADA 79687)**: The **actual** amount of power produced by wind farms, measured by sensors. This is our **Target Variable** (Ground Truth) that we want to predict.
    - **TOTAL WIND BASE SCHEDULE**: The **planned** amount of wind power that operators promised to generate ahead of time (Forecast).
    - **TOTAL WIND BASEPOINT**: The specific target level for wind generation set by grid controllers in real-time to balance the system.
    - **TOTAL BPA CONTROL AREA LOAD**: The total amount of electricity currently being used by homes, businesses, and industries in the region (Demand).
    - **TOTAL HYDRO GENERATION**: Electricity generated by dams and water turbines.
    - **TOTAL FOSSIL/BIOMASS GENERATION**: Electricity generated by burning fuels like natural gas, coal, or wood waste.
    - **TOTAL NUCLEAR GENERATION**: Electricity generated by nuclear power plants.
    - **NET INTERCHANGE**: The difference between energy flowing out to other regions and energy flowing in. It shows if the BPA grid is currently "exporting" or "importing" power.
    - **TOTAL SOLAR GENERATION / SCHEDULE / BASEPOINT**: These variables are the solar energy versions of the wind variables above. They track power from the sun, but are less critical for our specific Wind Ramp analysis.

**Stream B: Meteorological Data (NOAA/IEM)———API Download**

- **Source Format:** CSV files fetched via API.
- **Raw Structure:** **Long Format** .
  - Columns: `station`, `valid` (timestamp), `sknt` (speed), `drct` (direction).
- **Processed Structure:** Transformed via **Pivot** into **Wide Format**.
  - **Index:** `valid` (timestamp).
  - **Columns:** 
    - **station**: A unique ID for each weather station (e.g., "PDX" for Portland International Airport). It tells us the geographic location of the data.
    - **valid**: The exact timestamp of the observation, precise to the minute. This isused to align the weather data with the power grid data.
    - **tmpf**: Air temperature measured in degrees Fahrenheit. It shows how hot or coldthe environment is.
    - **dwpf**: Dew Point temperature in degrees Fahrenheit. It measures the amount ofmoisture or humidity in the air.
    - **drct**: Wind Direction measured in degrees (0-360°). It indicates where the wind isblowing from, which is important for understanding wind energy patterns.
    - **sknt**: Wind Speed measured in "Knots." This is the key metric that determines howmuch power a wind turbine can generate.
    - **mslp**: Mean Sea Level Pressure measured in millibars (mb) or hectopascals (hPa).It is used to analyze large-scale weather systems and pressure changes.
    



# Code 

The following Python script implements the authenticated data collection pipeline. It defines the API endpoints, handles pagination logic, and saves the raw data to the data/raw-data directory. Note that for security, API credentials are represented as placeholders.

In [ ]:
import pandas as pd
import numpy as np
import io
import requests
import os
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

In [11]:
# ================= Configuration =================
STATIONS = ["KSEA", "KPDX", "KGEG", "KEUG"]
START_DATE = "2022-01-01"
END_DATE = "2025-12-01"

OUTPUT_DIR = "../../data/raw-data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "BPA_Weather_IEM_Parallel.csv")

# ================= Core Function =================

def fetch_single_station(station, start, end):
    """
    Download ASOS data for a single station (used by the thread pool).
    """
    print(f"[{station}] Starting download...")

    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")

    base_url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
    params = {
        "station": station,
        "data": "all",
        "year1": s.year, "month1": s.month, "day1": s.day,
        "year2": e.year, "month2": e.month, "day2": e.day,
        "tz": "Etc/UTC",
        "format": "csv",
        "latlon": "no",
        "missing": "null",
    }

    try:
        r = requests.get(base_url, params=params, timeout=60)

        if r.status_code == 200:
            if "ERROR" in r.text[:50]:
                print(f"[{station}] Server returned an error.")
                return None

            df = pd.read_csv(io.StringIO(r.text), comment="#")
            print(f"[{station}] Completed: {len(df)} rows retrieved.")
            return df
        else:
            print(f"[{station}] Request failed with status code {r.status_code}.")
            return None

    except Exception as ex:
        print(f"[{station}] Exception occurred: {ex}")
        return None

# ================= Main Execution =================
if not os.path.exists(OUTPUT_DIR):
    try:
        os.makedirs(OUTPUT_DIR)
        print(f"Created directory: {OUTPUT_DIR}")
    except OSError:
        print(f"Failed to create directory {OUTPUT_DIR}. Saving to current directory.")
        OUTPUT_FILE = "BPA_Weather_IEM_Parallel.csv"

all_data_frames = []
print(f"=== Parallel download started ({START_DATE} to {END_DATE}) ===\n")
with ThreadPoolExecutor(max_workers=5) as executor:
    future_to_station = {
        executor.submit(fetch_single_station, station, START_DATE, END_DATE): station
        for station in STATIONS
    }
    for future in as_completed(future_to_station):
        station = future_to_station[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                all_data_frames.append(df)
        except Exception as exc:
            print(f"[{station}] Error retrieving result: {exc}")
if all_data_frames:
    print("\nMerging data...")
    
    final_df = pd.concat(all_data_frames, ignore_index=True)
    cols_needed = ['station', 'valid', 'tmpf', 'dwpf', 'drct', 'sknt', 'mslp']
    existing_cols = [c for c in cols_needed if c in final_df.columns]

    final_df = final_df[existing_cols]
    final_df['valid'] = pd.to_datetime(final_df['valid'])
    final_df.sort_values(by=['valid', 'station'], inplace=True)

    print("===== Completed =====")
    print(f"Total rows: {len(final_df)}")
    print(final_df.head())

    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nFile saved to: {OUTPUT_FILE}")
else:
    print("\nNo valid data downloaded.")

=== Parallel download started (2022-01-01 to 2025-12-01) ===

[KSEA] Starting download...
[KPDX] Starting download...
[KGEG] Starting download...
[KEUG] Starting download...


/var/folders/6f/mpj7nyk15wx2xll23dsk4zk80000gn/T/ipykernel_33148/2396599604.py:47: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(r.text), comment="#")


[KEUG] Completed: 429954 rows retrieved.


/var/folders/6f/mpj7nyk15wx2xll23dsk4zk80000gn/T/ipykernel_33148/2396599604.py:47: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(r.text), comment="#")


[KGEG] Completed: 427856 rows retrieved.


/var/folders/6f/mpj7nyk15wx2xll23dsk4zk80000gn/T/ipykernel_33148/2396599604.py:47: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(r.text), comment="#")


[KSEA] Completed: 429506 rows retrieved.
[KPDX] Completed: 425108 rows retrieved.

Merging data...
===== Completed =====
Total rows: 1712424
        station               valid  tmpf  dwpf   drct  sknt  mslp
0           EUG 2022-01-01 00:00:00   NaN   NaN   40.0   2.0   NaN
429954      GEG 2022-01-01 00:00:00   NaN   NaN  250.0   5.0   NaN
1287316     PDX 2022-01-01 00:00:00   NaN   NaN  120.0   9.0   NaN
857810      SEA 2022-01-01 00:00:00   NaN   NaN  340.0   5.0   NaN
1           EUG 2022-01-01 00:05:00   NaN   NaN    0.0   0.0   NaN

File saved to: ../../data/raw-data/BPA_Weather_IEM_Parallel.csv


In [7]:
def data_summary(df):
    """
    Print a concise dataset summary for reporting and validation.
    """
    print("=" * 40)
    print("       DATASET SUMMARY REPORT       ")
    print("=" * 40)

    # Dimensions
    print("\n1. Dimensions:")
    print(f"   - Rows:    {df.shape[0]:,}")
    print(f"   - Columns: {df.shape[1]}")

    # Temporal coverage
    if isinstance(df.index, pd.DatetimeIndex):
        start, end = df.index.min(), df.index.max()
        print("\n2. Temporal Coverage:")
        print(f"   - Start: {start}")
        print(f"   - End:   {end}")
        print(f"   - Duration: {end - start}")

    elif "timestamp" in df.columns or "Date/Time (UTC)" in df.columns:
        col = "timestamp" if "timestamp" in df.columns else "Date/Time (UTC)"
        try:
            ts = pd.to_datetime(df[col])
            print("\n2. Temporal Coverage:")
            print(f"   - Start: {ts.min()}")
            print(f"   - End:   {ts.max()}")
        except Exception:
            pass

    # Missing value analysis
    print("\n3. Missing Value Analysis:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        "Missing Count": missing,
        "Percentage (%)": missing_pct
    })

    missing_df = (
        missing_df[missing_df["Missing Count"] > 0]
        .sort_values(by="Percentage (%)", ascending=False)
    )

    if not missing_df.empty:
        print("   Columns with missing data:\n")
        print(missing_df.head(10).to_string())
        if len(missing_df) > 10:
            print(f"   ... and {len(missing_df) - 10} more columns.")
    else:
        print("   No missing values detected.")

    # Data types
    print("\n4. Data Types:")
    print(df.dtypes.value_counts().to_string())

    # Target variable summary (if available)
    target_col = "Power_MW"
    if target_col in df.columns:
        desc = df[target_col].describe()
        print(f"\n5. Target Variable Stats ({target_col}):")
        print(f"   - Mean:   {desc['mean']:.2f} MW")
        print(f"   - Std:    {desc['std']:.2f} MW")
        print(f"   - Min:    {desc['min']:.2f} MW")
        print(f"   - Max:    {desc['max']:.2f} MW")
        print(f"   - Median: {desc['50%']:.2f} MW")

    print("\n" + "=" * 40)
    print("           END OF REPORT            ")
    print("=" * 40)


# Load datasets
bpa_weather_data = pd.read_csv("../../data/raw-data/BPA_Weather_IEM_Parallel.csv")
print("BPA Weather Data:")
data_summary(bpa_weather_data)

wind_gen_data = pd.read_excel("../../data/raw-data/WindGenTotalLoadYTD_2022.xlsx", header=1)
print("Wind Generation Data:")
data_summary(wind_gen_data)


bpa weather data:
       DATASET SUMMARY REPORT       

1. Dimensions:
   - Rows:    1,712,424
   - Columns: 7

3. Missing Value Analysis:
   Columns with missing data:

      Missing Count  Percentage (%)
mslp        1575597       92.009748
dwpf        1545969       90.279569
tmpf        1545951       90.278517
drct          67953        3.968235
sknt          52697        3.077334

4. Data Types:
float64    5
object     2

           END OF REPORT            
Wind Gen Data:
       DATASET SUMMARY REPORT       

1. Dimensions:
   - Rows:    105,120
   - Columns: 13


/var/folders/6f/mpj7nyk15wx2xll23dsk4zk80000gn/T/ipykernel_33148/54488910.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df[col])



2. Temporal Coverage:
   - Start: 2022-01-01 08:00:00
   - End:   2023-01-01 07:55:00

3. Missing Value Analysis:
   Columns with missing data:

                                                                             Missing Count  Percentage (%)
TOTAL WIND BASE SCHEDULE (FORECAST) IN BPA CONTROL AREA (MW; SCADA 187517            50107       47.666476
TOTAL SOLAR BASE SCHEDULE (FORECAST) IN BPA CONTROL AREA (MW; SCADA 187513)          50107       47.666476
TOTAL SOLAR Basepoint (FORECAST) IN BPA CONTROL AREA (MW; SCADA 177165)                 23        0.021880
TOTAL WIND BASEPOINT (FORECAST) IN BPA CONTROL AREA (MW; SCADA 103349                   12        0.011416
TOTAL WIND GENERATION  IN BPA CONTROL AREA (MW; SCADA 79687)                            12        0.011416
TOTAL BPA CONTROL AREA LOAD (MW; SCADA 45583)                                           12        0.011416
TOTAL HYDRO GENERATION (MW; SCADA 79682)                                                12        0.01141

# Summary and Interpretation of Results

  ### **1. Weather Data Stream (NOAA/IEM Source)**

  - **Volume:** ~1.71 million rows (Raw "Long Format").
  - **Key Strengths:**
    - **Kinetic Features:** **Wind Speed (`sknt`)** and **Direction (`drct`)** show excellent availability with **<4% missingness**, confirming the dataset is viable for wind dynamics modeling.
  - **Key Limitations:**
    - **Thermodynamic Features:** Variables like **Pressure (`mslp`)** and **Temperature (`tmpf`)** have **>90% missingness**, indicating coarser reporting frequencies. These columns will likely be dropped in the cleaning phase.

  ### **2. Grid Generation Data Stream (BPA Source)**

  - **Volume:** 105,120 rows (2022-2023), perfectly matching the expected 5-minute sampling frequency.
  - **Key Strengths:**
    - **Target Integrity:** The primary target, **`TOTAL WIND GENERATION`**, is **99.99% complete** , providing a solid ground truth.
  - **Key Limitations:**
    - **Forecast Data:** "Base Schedule" columns are nearly **48% missing**, likely due to reporting inconsistencies. However, this does not impact our ability to model actual generation.

{{< include closing.qmd >}} 